# 2D Annular Cavity Void Regression: Epistemic Uncertainty with DP-BNNs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Paper:** *Dirichlet Process Bayesian Neural Networks for Epistemic Uncertainty and Deep Exploration* (TMLR 2026)  
**Author:** Sumit Vashishtha  

---

### 1. Pedagogical Overview: Why is Epistemic Uncertainty Hard?
In deep learning, predictive uncertainty has two distinct components:
1. **Aleatoric Uncertainty:** Inherent noise in the observations $\epsilon \sim \mathcal{N}(0, \sigma_{\text{aleatoric}}^2)$ that cannot be reduced with more data.
2. **Epistemic Uncertainty:** Uncertainty in the model parameters or function hypothesis due to a lack of data in unexplored or out-of-distribution regions.

Standard Bayesian Neural Networks (BNNs) attempt to quantify epistemic uncertainty by placing Gaussian priors over neural network weights $w \sim \mathcal{N}(0, I)$. However, in deep networks with millions of parameters:
* **Reverse-KL Variational Inference (BBB / MC-Dropout)** severely underestimates variance and collapses into deterministic predictions on unvisited voids.
* **Deep Ensembles** maintain epistemic diversity by training $K$ distinct networks with different random weight initializations, but this scales memory, compute, and optimizer buffers by $\mathcal{O}(K)$.

### 2. The DP-BNN Paradigm Shift: Priors in Data Space
Rather than placing intractable priors over opaque weight parameters, **Dirichlet Process Bayesian Neural Networks (DP-BNNs)** place a nonparametric Dirichlet Process prior directly on the **data-generating distribution** $G^* \sim \mathrm{DP}(\alpha, F_0)$ in experience space:
$$G^* \mid \mathcal{D}_n \sim \mathrm{DP}\left(\alpha + n, \, \frac{\alpha F_0 + \sum_{i=1}^n \delta_{z_i}}{\alpha + n}\right)$$

By sampling random probability measures $P_j$ from this posterior using the constructive stick-breaking representation derived by **Vashishtha & Maillard (2025)**, each posterior draw $P_j$ pulls a neural network toward a distinct hypothesis in unobserved regions.

In this notebook, we reproduce the **2D Annular Cavity Void benchmark** (Section 4 & Figure 1 of the paper), demonstrating how DP-BNNs form a sharp, isotropic **epistemic bubble** in an unobserved void where variational BNNs fail.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Set random seeds for exact reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")


### 3. Generating the Annular Dataset with an Unobserved Central Cavity
We generate $n = 300$ spatial training points $x = (x_1, x_2) \in [-2.5, 2.5]^2$ lying strictly within the annular ring:
$$\text{Radius } r = \|x\|_2 \in [1.3, 2.45]$$
The ground truth function is $y = \cos(1.5\pi r) + \epsilon$ with aleatoric noise $\sigma_{\text{aleatoric}} = 0.30$.  
Crucially, the central disc:
$$r = \|x\|_2 \le 1.0$$
contains **zero training points** (the cavity void). A reliable epistemic model must detect this void and express high uncertainty!


In [ ]:
# Generate annular dataset
pts = []
while len(pts) < 300:
    p = np.random.uniform(-2.5, 2.5, size=2)
    r = np.linalg.norm(p)
    if 1.3 <= r <= 2.45:
        pts.append(p)

X_train = np.array(pts, dtype=np.float32)
r_train = np.linalg.norm(X_train, axis=1, keepdims=True)
clean_y = np.cos(1.5 * np.pi * r_train)
noise = np.random.normal(0.0, 0.30, size=clean_y.shape).astype(np.float32)
y_train = (clean_y + noise).astype(np.float32)

print(f"Training data shape: X = {X_train.shape}, y = {y_train.shape}")
print(f"Min radius: {r_train.min():.2f}, Max radius: {r_train.max():.2f}")


### 4. Neural Network Architecture
We define a standard 2-layer MLP (2 inputs $\to$ 64 $\to$ 64 $\to$ 1 output) with ReLU activations.


In [ ]:
class MLP2D(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x)

# Quick sanity check
test_net = MLP2D()
print(test_net)


### 5. Constructive Stick-Breaking Posterior Sampling (Vashishtha & Maillard, 2025)

Many implementations take a heuristic shortcut by sampling weights from a finite symmetric Dirichlet distribution:
$$\mathbf{w} \sim \mathrm{Dir}\left(1, \dots, 1, \, \frac{\alpha}{M}, \dots, \frac{\alpha}{M}\right)$$
While related, this is **not** the true stick-breaking process!

Here, we implement the **exact constructive stick-breaking representation** derived by **Vashishtha & Maillard (2025)** (Section 3 of the paper):
1. **Prior Mass Allocation:** Under Ferguson's conjugacy, the total mass allocated to the prior base measure follows a Beta distribution:
   $$W_{\text{prior}} \sim \mathrm{Beta}(\alpha, n), \quad W_{\text{data}} = 1 - W_{\text{prior}}$$
2. **Empirical Stick-Breaking:** Empirical observations receive sequential stick-breaking weights:
   $$V_i \sim \mathrm{Beta}(1, \alpha + i), \quad w_i^{\text{emp}} = V_i \prod_{j=i+1}^n (1 - V_j)$$
3. **Prior Base Measure Stick-Breaking:** The $T$ synthetic prior atoms from $F_0$ receive Sethuraman stick-breaking weights:
   $$V_t \sim \mathrm{Beta}(1, \alpha), \quad q_t = V_t \prod_{l < t} (1 - V_l)$$
4. **Combined Random Measure:**
   $$\mathbf{w} = \Big[(1 - W_{\text{prior}}) \frac{\mathbf{w}^{\text{emp}}}{\sum w_k^{\text{emp}}}, \quad W_{\text{prior}} \frac{\mathbf{q}}{\sum q_l}\Big]$$


In [ ]:
def sample_vashishtha_maillard_weights(n_emp, T_prior, alpha, rng=np.random):
    """
    Exact constructive stick-breaking random measure sampler (Vashishtha & Maillard, 2025).
    """
    # 1. Total mass allocated to prior base measure
    W_prior = float(rng.beta(alpha, float(n_emp)))
    W_data = 1.0 - W_prior
    
    # 2. Sequential stick-breaking for empirical observations: V_i ~ Beta(1, alpha + i)
    i_arr = np.arange(1, n_emp + 1, dtype=np.float32)
    V_emp = rng.beta(1.0, alpha + i_arr).astype(np.float32)
    U_emp = 1.0 - V_emp
    suffix_U = np.ones(n_emp + 1, dtype=np.float32)
    suffix_U[:-1] = np.cumprod(U_emp[::-1])[::-1]
    
    w_emp_raw = np.zeros(n_emp, dtype=np.float32)
    if n_emp > 1:
        w_emp_raw[:-1] = V_emp[:-1] * suffix_U[1:-1]
    w_emp_raw[-1] = V_emp[-1]
    w_emp = W_data * (w_emp_raw / w_emp_raw.sum())
    
    # 3. Sethuraman stick-breaking for prior base measure atoms: V_t ~ Beta(1, alpha)
    V_prior = rng.beta(1.0, max(alpha, 1e-3), size=T_prior).astype(np.float32)
    V_prior[-1] = 1.0  # Truncation at T_prior
    cum_prod_p = np.cumprod(1.0 - V_prior)
    q_p = np.empty_like(V_prior)
    q_p[0] = V_prior[0]
    q_p[1:] = V_prior[1:] * cum_prod_p[:-1]
    w_prior = W_prior * (q_p / q_p.sum())
    
    # 4. Combined discrete measure weights
    return np.concatenate([w_emp, w_prior]), W_prior

K_MODELS = 10
M_PRIOR = 50
ALPHA_DP = 10.0
EPOCHS = 250
LR = 0.01

models = []

for k in range(K_MODELS):
    # Sample synthetic prior atoms from F_0 = U([-2.5, 2.5]^2) x N(0, 1.5^2)
    px = np.random.uniform(-2.5, 2.5, size=(M_PRIOR, 2)).astype(np.float32)
    py = np.random.normal(0.0, 1.5, size=(M_PRIOR, 1)).astype(np.float32)
    
    all_X = np.vstack([X_train, px])
    all_y = np.vstack([y_train, py])
    
    # Exact Vashishtha & Maillard (2025) stick-breaking weights
    weights, w_p = sample_vashishtha_maillard_weights(len(X_train), M_PRIOR, ALPHA_DP)
    scaled_w = weights * len(all_X)
    
    t_X = torch.tensor(all_X, dtype=torch.float32).to(device)
    t_y = torch.tensor(all_y, dtype=torch.float32).to(device)
    t_w = torch.tensor(scaled_w, dtype=torch.float32).unsqueeze(1).to(device)
    
    net = MLP2D().to(device)
    optimizer = optim.Adam(net.parameters(), lr=LR)
    
    for epoch in range(EPOCHS):
        optimizer.zero_grad()
        loss = (t_w * (net(t_X) - t_y)**2).mean()
        loss.backward()
        optimizer.step()
        
    net.eval()
    models.append(net)
    print(f"Trained posterior model {k+1}/{K_MODELS} (Prior Mass W_prior = {w_p:.3f})")


### 6. Evaluating Epistemic Uncertainty via BALD on a 2D Grid
We evaluate the ensemble of $J=10$ models across a $50 \times 50$ 2D spatial grid.
We compute epistemic uncertainty using **Bayesian Active Learning by Disagreement (BALD)** (Houlsby et al., 2011):
$$\text{BALD}(x) = \frac{1}{2} \ln\left(1 + \frac{\sigma_{\text{epistemic}}^2(x)}{\sigma_{\text{aleatoric}}^2}\right)$$
where $\sigma_{\text{epistemic}}^2(x) = \mathrm{Var}_{j}(f_{\theta^{(j)}}(x))$ is the variance across posterior models, and $\sigma_{\text{aleatoric}} = 0.30$ is the known observation noise.


In [ ]:
# Create 2D evaluation grid
grid_size = 50
x1_lin = np.linspace(-2.5, 2.5, grid_size)
x2_lin = np.linspace(-2.5, 2.5, grid_size)
X1, X2 = np.meshgrid(x1_lin, x2_lin)
grid_pts = np.stack([X1.ravel(), X2.ravel()], axis=1).astype(np.float32)
t_grid = torch.tensor(grid_pts, dtype=torch.float32).to(device)

# Posterior predictions across all J=10 models
with torch.no_grad():
    all_preds = torch.stack([m(t_grid) for m in models], dim=0).squeeze(-1).cpu().numpy()

mean_pred = all_preds.mean(axis=0).reshape(grid_size, grid_size)
var_pred = all_preds.var(axis=0).reshape(grid_size, grid_size)
bald = 0.5 * np.log(1.0 + var_pred / (0.30**2))

# Compute void-to-data contrast ratio
grid_r = np.linalg.norm(grid_pts, axis=1)
void_mask = grid_r <= 1.0
data_mask = (grid_r >= 1.3) & (grid_r <= 2.45)

void_bald = bald.ravel()[void_mask].mean()
data_bald = bald.ravel()[data_mask].mean()
contrast_ratio = void_bald / data_bald

print(f"Mean BALD in Cavity Void (r <= 1.0):      {void_bald:.4f} nats")
print(f"Mean BALD on Data Ring (1.3 <= r <= 2.45): {data_bald:.4f} nats")
print(f"Void-to-Data Contrast Ratio:              {contrast_ratio:.2f}x (similar to paper range)")


### 7. Visualizing the Epistemic Bubble (Figure 1 in Paper)
Notice how DP-BNN creates a sharp, circular **epistemic bubble** in the unobserved central void ($r \le 1.0$)!


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Training Data
circle_void = plt.Circle((0, 0), 1.0, color='red', fill=False, linestyle='--', linewidth=2, label='Cavity Void (r <= 1.0)')
circle_inner = plt.Circle((0, 0), 1.3, color='black', fill=False, linestyle=':', linewidth=1.5, label='Annulus Inner Bound (r=1.3)')
circle_outer = plt.Circle((0, 0), 2.45, color='black', fill=False, linestyle='-', linewidth=1.5, label='Annulus Outer Bound (r=2.45)')

axes[0].scatter(X_train[:, 0], X_train[:, 1], c=y_train[:, 0], cmap='viridis', s=20, alpha=0.8, edgecolors='none')
axes[0].add_patch(circle_void)
axes[0].add_patch(circle_inner)
axes[0].add_patch(circle_outer)
axes[0].set_xlim(-2.5, 2.5)
axes[0].set_ylim(-2.5, 2.5)
axes[0].set_aspect('equal')
axes[0].set_title("(a) Training Points in Annular Ring\n(Unobserved Void for r <= 1.0)", fontsize=12)
axes[0].legend(loc='upper right', fontsize=8)

# Panel 2: Posterior Mean
im1 = axes[1].contourf(X1, X2, mean_pred, levels=50, cmap='Spectral_r')
circle_void2 = plt.Circle((0, 0), 1.0, color='red', fill=False, linestyle='--', linewidth=2)
axes[1].add_patch(circle_void2)
axes[1].set_xlim(-2.5, 2.5)
axes[1].set_ylim(-2.5, 2.5)
axes[1].set_aspect('equal')
axes[1].set_title(r"(b) DP-BNN Posterior Mean $\mu(x)$", fontsize=12)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# Panel 3: Epistemic Uncertainty (BALD)
im2 = axes[2].contourf(X1, X2, bald, levels=50, cmap='magma')
circle_void3 = plt.Circle((0, 0), 1.0, color='cyan', fill=False, linestyle='--', linewidth=2, label='Cavity Void')
axes[2].add_patch(circle_void3)
axes[2].set_xlim(-2.5, 2.5)
axes[2].set_ylim(-2.5, 2.5)
axes[2].set_aspect('equal')
axes[2].set_title(f"(c) Epistemic Bubble (BALD)\nVoid/Data Contrast: {contrast_ratio:.1f}x", fontsize=12)
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


### 8. Pedagogical Takeaways
1. **Priors in Data Space Work:** In parametric BNNs (variational Bayes), reverse-KL projections force the approximate posterior variance to collapse to zero wherever gradients lack data support. In contrast, DP-BNNs place prior mass directly on the data distribution. In the void, each random measure $P_j$ samples different target values from $F_0$, pulling each network toward a different functional hypothesis and naturally inflating epistemic variance.
2. **Isotropic Uncertainty:** Unlike deep ensembles whose variance can leak irregularly across directions, the DP prior base measure $F_0 = \mathcal{U}([-2.5, 2.5]^2) \times \mathcal{N}(0, \sigma_0^2)$ provides uniform coverage across all angles, forming a clean, isotropic circular bubble.
3. **Reference:** For the mathematical stick-breaking random measure derivation, see:
   * **Vashishtha, S., & Maillard, O.-A. (2025).** *Leveraging priors on distribution functions for multi-arm bandits.* Reinforcement Learning Journal, 6:1600–1623.
